# Explore model data with observational data from Barcelona using ICOS Python Libraries 

This example uses a foundational library called icoscp_core which can be used to access ICOS data that are previewable in the ICOS Data Portal. 

The simulations were conducted using [Weather Research and Forecasting with chemistry (WRF-Chem) v4.3.3 model with greenhouse gas passive tracer option](https://www.bgc-jena.mpg.de/archived/bgc-systems/bgc-systems/uploads/Wrf-ghg/Technical%20Reports%202011%20Beck.pdf) to assess the source attribution of atmospheric CO₂ and quantify the contributions from biogenic and anthropogenic sources.

The model was configured with three nested domains, with horizontal resolutions of 9, 3, and 1 km, respectively. Each domain comprised 45 vertical layers extending up to 50 hPa. Simulations covered the full year of 2023, and results presented here are at 1 km. 
Input emission inventories:(1) a high-resolution anthropogenic CO₂ emission inventory derived from the High-Elective Resolution Modelling Emission System delta version ([HERMES_∆](https://github.com/BSC-ES/HERMES_Delta))  and (2) biogenic CO₂ fluxes estimated using the Vegetation Photosyntheis and Respiration Model([VPRM](https://zenodo.org/records/10782550)). The initial and boundary condition data for the CO₂  concentrations were obtained from the [CAMS global inversion-optimised greenhouse gas fluxes and concentrations](https://ads.atmosphere.copernicus.eu/datasets/cams-global-greenhouse-gas-inversion?tab=overview))  

General information on all ICOS Carbon Portal Python libraries can be found on our [help pages](https://icos-carbon-portal.github.io/pylib/). Documentation of the icoscp_core library, including information on running it locally, can also be found on [PyPI.org](https://pypi.org/project/icoscp_core/).

Prepared by "Qing Luo" qing.luo@uab.cat,"Yi Wang" yi.wang@mgeo.lu.se, "Vanessa Monteiro" Vanessa.Monteiro@uab.cat

# Load libraries

In [ ]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import requests
from icoscp_core.cities import meta as citymeta, data as citydata
from icoscp_core.cities import auth
import numpy as np
import folium
from scripts import obs_nc_explorer
Validate = obs_nc_explorer.Validate
MAPTS = obs_nc_explorer.MAPTS

# Access observational data

In [ ]:
# all stations
stations = citymeta.list_stations()

# extract stations in Spain
es_stations = [s for s in stations if s.country_code == 'ES']

In [ ]:
# convert station list to a pandas data frame
df = pd.DataFrame(es_stations)
df

Note: first uri in the list above is for the entire Barcelona domain, not an observational station

In [ ]:
# A list for all observational stations
ES_OBS = [
    s for s in stations
    if s.country_code == 'ES' and 'Barcelona_City' not in s.uri
]
pd.DataFrame(ES_OBS)

In [ ]:
# List data objects from all stations
dobj_list = citymeta.list_data_objects(station=ES_OBS)
pd.DataFrame(dobj_list)

In [ ]:
# Compile data from all stations
# Quality Control
station_data = {}
station_meta = {}

for dobj in dobj_list:
    uri = dobj.uri
    station = dobj.station_uri.split("BCN_", 1)[-1]

    dobj_meta = citymeta.get_dobj_meta(uri)
    dobj_cols = citydata.get_columns_as_arrays(dobj_meta)
    dobj_data = pd.DataFrame(dobj_cols)

    # QC
    dobj_data = dobj_data[dobj_data["Flag"] == "O"].copy()

    # Set TIMESTAMP as index
    dobj_data["TIMESTAMP"] = pd.to_datetime(dobj_data["TIMESTAMP"])
    dobj_data = dobj_data.set_index("TIMESTAMP")

    station_data[station] = dobj_data
    station_meta[station] = dobj_meta

    # # ----------------------------------------
    # # create a combined metadata dictionary including elevation and sampling height for the model interpolation
    # # ----------------------------------------

    # station_obj = next(
    #     (
    #         s for s in ES_OBS
    #         if s.name == station
    #         or s.id == station
    #     ),
    #     None
    # )
    
    # if station_obj is None:
    #     print(f"Warning: metadata not found for {station}")
    #     continue
        
    # station_meta[station] = {

    #     "lat": float(station_obj.lat),

    #     "lon": float(station_obj.lon),

    #     "elevation": float(station_obj.elevation),

    #     "sampling_height": float(dobj.sampling_height),
    # }

In [ ]:
print(station_data["Fabra"])

# Access model netCDF data

## Authentication
You will need to have an account at https://cpauth.icos-cp.eu/login/

In [ ]:
# Uncomment to authenticate
# Only need to execute this cell once
# Comment out after successful authentication
# auth.init_config_file()

In [ ]:
# Find your token and validate if your token is valid
token = auth.get_token()
if token.is_fresh(): print('Token is valid')

In [ ]:
session = requests.Session()
session.headers.update({
    'Authorization': 'Bearer '+auth.get_token().cookie_value,
})

## Helper functions

In [ ]:
# Helper function for url construction and pydap access for one single dataset
def getSingleDataSet(dobj_uri:str):
    objId = dobj_uri.split("/")[-1]
    url = 'dap2://opendap.icos-cp.eu/' + objId
    return xr.open_dataset(url, engine='pydap', backend_kwargs={'session': session})

In [ ]:
# Alternative way for open and combine multiple files
def getMultiDataSet(files:list, **kwargs):
    objIds = [f'dap2://opendap.icos-cp.eu/{f.split("/")[-1]}' for f in files]
    return xr.open_mfdataset(objIds, engine='pydap', backend_kwargs={'session': session}, data_vars='all')

## Get  metadata

In [ ]:
BCN = [s for s in citymeta.list_stations() if 'Barcelona_City' in s.uri]
pd.DataFrame(BCN)

In [ ]:
# List data objects from this station
wrf_dobj_list = citymeta.list_data_objects(station=BCN)
wrf_dobj_list

## Access one model NetCDF file

In [ ]:
# Here we only access the first data object of this station
wrf_dobj_uri = wrf_dobj_list[0].uri
wrf_dobj_uri

In [ ]:
# read in dataset as a xarray dataset
#Click document logo to see the attributes of each variable; and data logo to see the data structure of each variable

ds = getSingleDataSet(wrf_dobj_uri)

In [ ]:
ds

In [ ]:
# ======================================================
# get Latitude / Longitude
# ======================================================
lat = ds["lat"].values
lon = ds["lon"].values
print("\nSpatial Information")
print("-" * 70)

print(f"Latitude range : {np.nanmin(lat):.2f} to {np.nanmax(lat):.2f}")
print(f"Longitude range: {np.nanmin(lon):.2f} to {np.nanmax(lon):.2f}")

In [ ]:
# here show the modelled area in a map
# Domain boundary (clockwise)
boundary = []
# Bottom edge (west -> east)
boundary.extend(zip(lat[0, :], lon[0, :]))
# Right edge (south -> north)
boundary.extend(zip(lat[1:, -1], lon[1:, -1]))
# Top edge (east -> west)
boundary.extend(zip(lat[-1, -2::-1], lon[-1, -2::-1]))
# Left edge (north -> south)
boundary.extend(zip(lat[-2:0:-1, 0], lon[-2:0:-1, 0]))
# Close the polygon
boundary.append(boundary[0])

m = folium.Map(
    location = [lat.mean(), lon.mean()],
    tiles = "OpenStreetMap", 
    zoom_start = 10
)
folium.Polygon(
    locations=boundary,
    color="red",
    weight=3,
    fill=True,
    fill_color="red",
    fill_opacity=0.08,
    tooltip="WRF d03 Domain"
).add_to(m)
m

In [ ]:
# print("\nFirst 10 values")
# print("=" * 60)
# #WRF uses terrain-following/hybrid vertical coordinate
# #eta->1 near the surface   eta->0 near the model top  50hPa
# print("\neta:")
# print(ds["eta"].values[:10]) 

# print("\nZ_sfc:")
# print(ds["Z_sfc"].values.flatten()[:10]) #surface height/terrain height

# print("\nZ_e at first time and first grid point:")
# print(ds["Z_e"].isel(time=0, south_north=0, west_east=0).values[:10]) #geometric height/above sea level height

### Validate the model using observations

Why validate the WRF-Chem CO₂ simulation with observations?

The purpose of model validation is to evaluate whether the simulated CO₂ field reproduces the temporal behaviour observed at real measurement stations.

A model can produce physically reasonable CO₂ concentrations, but this does not  mean that it correctly represents the temporal variability at a specific urban location. Observations therefore provide an independent reference for assessing the model performance. The model output is sampled at the nearest WRF-Chem grid cell and vertically interpolated to the actual observation inlet height, so that the model and observation represent comparable atmospheric conditions.


Note: 

1)We compare the observed and simulated CO₂ concentrations at different temporal scales:Hourly,Daily mean and Monthly-averaged diurnal cycle

2)The model CO₂ is separated into:Background CO₂, Biogenic CO₂, Anthropogenic CO₂

This allows us not only to ask whether the total CO₂ concentration is realistic, but also to understand which source contributes to the simulated variability. 
In the model, all concentrations need to be positive, we added 400 ppm manually for tracer2(biosphere) and tracer3(anthropogenic) during importing the background concentrations. Thus 400 ppm is substracted from them in the postprocessing. 

Check more in obs_nc_explorer.py, in Class Validate



In [ ]:
mpvd = Validate(ds,station_data,station_meta)
mpvd.show()

### How to interpret the above figures?
Take one station ICTA for example:

1) Hourly CO₂ time series

   The model reproduces part of the temporal evolution, but the simulated curves generally have a smaller amplitude than the observations.

    The separation between the model curves helps identify the source responsible for the simulated enhancement: the background component remains relatively stable;the biogenic contribution is comparatively small;the anthropogenic component dominates the diurnal change.

    The hourly comparison gives:R = 0.67,RMSE = 21.77 ppm,MB = −16.03 ppm. These indicate that the model captures a meaningful part of the observed hourly temporal variability, but it also underestimates the observed CO₂ concentration on average, as indicated by the negative mean bias.

2) Monthly-averaged diurnal cycle

   show the average CO₂ behaviour for each hour of the day over the entire month and help to understand the modelled source contributions. Observed CO₂ is relatively high during the night and early morning, reaching a maximum around the morning hours.The model reproduces the broad shape of this daily cycle, although the simulated amplitude is much smaller than observed. Biogenic component serves as co2 source during the night while as slight co2 sink during the day.

3) Daily mean CO₂ time series

   The daily mean figure removes much of the short-term noise seen in the hourly series and focuses on the day-to-day evolution.The model  captures some monthly-scale temporal structure, but it does not reproduce the magnitude of several high-CO₂ days (for example, the peak at 10th Dec).The background CO₂ changes only slightly during the month, indicating that most of the modelled enhancement comes from the local source contributions rather than large changes in the background field.
   
4) Daily mean CO₂ source contributions

   identify which CO₂ sources are responsible for the modelled enhancement at ICTA on each day. It shows the anthropogenic contribution is clearly larger than the biogenic contribution for most days.The largest anthropogenic contribution occurs near the end of December, where it rises dramatically.The biogenic contribution remains relatively small, generally only a few ppm.


### Explore geospatial distribution of concentrations and time series at a chosen location
Why geospatial distribution?

The temporal evolution of CO₂ tells us when concentrations change, but the geospatial distribution tells us where CO₂ is enhanced and how the spatial pattern relates to emissions, vegetation, terrain, and atmospheric transport.

We therefore examine the spatial distribution of the simulated CO₂ field over the simulated domain. The model can be viewed for different temporal scales (hourly,daily, monthly) and for different CO₂ components, including background, biogenic, anthropogenic, and total CO₂.

Because WRF uses a terrain-following η coordinate, the corresponding physical height Z_e varies across the domain. We show the mean and range of Z_e for the selected level. When a grid cell is selected, it also reports the local Z_e, together with the grid index, latitude, longitude, and η value.

The maps provide an overview of where CO₂ concentrations are enhanced. To understand how the simulated CO₂ field behaves dynamically, we also need to examine its temporal evolution at a specific location.

Here we link the spatial and temporal views: users can select a grid cell directly on the WRF domain, and the model CO₂ time series at that location can then be examined while changing the time and vertical η level. It provides the background, biogenic, anthropogenic, and total CO₂ components.

Note: time series plot will take some time to update

In [ ]:
viewer = MAPTS(
    ds,
    offset=400,
    height=600,
    zoom=8,
    cache_cells=3,#increase cache_cells if you often revisit several grid cells, or reduce it to 1 to minimize cached memory.
).show()

## Access multiple NetCDF files

In [ ]:
dsall = getMultiDataSet([wrf_dobj.uri for wrf_dobj in wrf_dobj_list])

In [ ]:
dsall